# 06 - Treinar XGBoost

Classificador para prever WIN/LOSS com foco em **Precision**.

- TimeSeriesSplit (evitar look-ahead bias)
- Threshold de probabilidade > 65%
- Métricas: Precision, Sharpe Ratio

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    precision_score,
    roc_auc_score
)
import xgboost as xgb

plt.style.use('dark_background')
DATA_DIR = Path('../data')
MODEL_DIR = Path('../models')

In [ ]:
# Carregar features
with open(DATA_DIR / 'features.pkl', 'rb') as f:
    data = pickle.load(f)

X = data['X_signal']
y = data['y']

print(f'X shape: {X.shape}')
print(f'y: {y.sum()} WIN / {(~y.astype(bool)).sum()} LOSS')

## 1. TimeSeriesSplit (Evitar Look-ahead Bias)

In [ ]:
# Split temporal - NUNCA shuffle aleatório!
tscv = TimeSeriesSplit(n_splits=5)

print('=== TimeSeriesSplit (5 folds) ===')
for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
    print(f'Fold {i+1}: Train {len(train_idx)} | Test {len(test_idx)}')

## 2. Treinar XGBoost com Cross-Validation

In [ ]:
# Parâmetros XGBoost
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 4,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
}

# Cross-validation temporal
cv_results = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # Treinar
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, verbose=False)
    
    # Predições com probabilidade
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Aplicar threshold 65%
    y_pred_65 = (y_proba >= 0.65).astype(int)
    
    # Métricas
    precision = precision_score(y_test, y_pred_65, zero_division=0)
    trades_taken = y_pred_65.sum()
    auc = roc_auc_score(y_test, y_proba)
    
    cv_results.append({
        'fold': fold + 1,
        'precision': precision,
        'trades': trades_taken,
        'auc': auc,
    })
    
    print(f'Fold {fold+1}: Precision={precision:.2%} | Trades={trades_taken} | AUC={auc:.3f}')

cv_df = pd.DataFrame(cv_results)
print(f'\n=== Média ===')
print(f'Precision: {cv_df["precision"].mean():.2%}')
print(f'AUC: {cv_df["auc"].mean():.3f}')

## 3. Treinar Modelo Final

In [ ]:
# Treinar com todos os dados
final_model = xgb.XGBClassifier(**params)
final_model.fit(X, y, verbose=False)

print('Modelo final treinado!')

## 4. Feature Importance

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
importance.plot(kind='barh', x='feature', y='importance', ax=ax, color='cyan', legend=False)
ax.set_xlabel('Importance')
ax.set_title('Feature Importance - XGBoost')
plt.tight_layout()
plt.show()

print('=== Feature Importance ===')
print(importance.sort_values('importance', ascending=False))

## 5. Análise de Threshold

In [ ]:
# Testar diferentes thresholds
y_proba_all = final_model.predict_proba(X)[:, 1]

thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
threshold_results = []

for thresh in thresholds:
    y_pred = (y_proba_all >= thresh).astype(int)
    trades = y_pred.sum()
    if trades > 0:
        precision = precision_score(y, y_pred)
        win_rate = y[y_pred == 1].mean() * 100
    else:
        precision = 0
        win_rate = 0
    
    threshold_results.append({
        'threshold': thresh,
        'trades': trades,
        'precision': precision,
        'win_rate': win_rate,
    })

thresh_df = pd.DataFrame(threshold_results)
print(thresh_df)

In [ ]:
# Visualizar trade-off
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision vs Threshold
axes[0].plot(thresh_df['threshold'], thresh_df['precision'], 'o-', color='lime')
axes[0].axhline(0.70, color='red', linestyle='--', label='Target 70%')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision vs Threshold')
axes[0].legend()

# Trades vs Threshold
axes[1].plot(thresh_df['threshold'], thresh_df['trades'], 'o-', color='orange')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Número de Trades')
axes[1].set_title('Trades vs Threshold')

plt.tight_layout()
plt.show()

## 6. Salvar Modelo

In [ ]:
# Salvar modelo e configurações
model_data = {
    'model': final_model,
    'threshold': 0.65,
    'features': list(X.columns),
    'cv_results': cv_df,
}

with open(MODEL_DIR / 'xgboost_v1.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print(f'Modelo salvo em: {MODEL_DIR / "xgboost_v1.pkl"}')

## 7. Resumo

- **Modelo**: XGBoost Classifier
- **Threshold**: 65% probabilidade
- **Validação**: TimeSeriesSplit (5 folds)
- **Foco**: Precision > Recall (Sniper mode)